<a href="https://colab.research.google.com/github/Arobnett/HDX-sources-and-more-API-connection/blob/main/notebooks/04_eda_open_source.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 EDA Open Source

Rebuilds the validated public Gold feature table, generates reproducible EDA reports and visuals, and writes a concise findings summary for downstream feature engineering and modeling.

**Scope:** public predictors only. Restricted target variables are introduced later in Databricks.

## Technical references

- pandas descriptive statistics: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html
- pandas missing data: https://pandas.pydata.org/docs/user_guide/missing_data.html
- pandas groupby: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html
- pandas correlation: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html
- scikit-learn preprocessing concepts: https://scikit-learn.org/stable/modules/preprocessing.html
- scikit-learn common pitfalls / leakage: https://scikit-learn.org/stable/common_pitfalls.html

In [ ]:
from pathlib import Path  # Work with repository and output paths.
import importlib.util  # Check optional dependencies.
import os  # Change the active working directory.
import subprocess  # Clone or refresh the GitHub repository.
import sys  # Use the current Python interpreter and module path.

REPO_URL = "https://github.com/Arobnett/HDX-sources-and-more-API-connection.git"  # Public repository.
PROJECT_ROOT = Path("/content/HDX-sources-and-more-API-connection")  # Standard Colab clone location.

if not PROJECT_ROOT.exists():  # Clone the repository in a fresh Colab runtime.
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:  # Refresh an existing clone to the latest main branch.
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", "main"], check=True)

if importlib.util.find_spec("pycountry") is None:  # Install the ISO reference library when absent.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycountry"], check=True)

os.chdir(PROJECT_ROOT)  # Make repository-relative paths deterministic.
sys.path.insert(0, str(PROJECT_ROOT / "src"))  # Make reusable pipeline modules importable.
print(f"Repository ready: {PROJECT_ROOT}")

## Rebuild and validate the public Gold layer

This keeps the EDA reproducible from the current version-controlled Silver inputs.

In [ ]:
from cleaning import clean_silver_directory  # Rebuild source-specific Silver features.
from geographic_keys import validate_clean_directory  # Enforce strict geographic keys.
from feature_assembly import assemble_feature_table  # Build the public Gold feature table.
from paths import CLEAN_DIR, CLEAN_REPORTS_DIR, OUTPUTS_DIR  # Canonical output paths.

clean_summary, clean_rejects = clean_silver_directory()  # Rebuild model-ready Silver files.
geo_summary = validate_clean_directory(CLEAN_DIR, CLEAN_REPORTS_DIR)  # Validate/canonicalize geography.
base_features, assembly_report, feature_catalog = assemble_feature_table()  # Assemble validated Gold.

KEY_COLUMNS = ["iso3", "country", "year", "month"]  # Define the modeling grain.
assert not base_features.empty, "Gold table is empty."  # Stop on unusable output.
assert not base_features.duplicated(KEY_COLUMNS).any(), "Duplicate Gold modeling keys detected."  # Enforce uniqueness.

print(f"Gold rows: {len(base_features):,}")
print(f"Gold columns: {len(base_features.columns):,}")
print(f"Countries: {base_features['iso3'].nunique():,}")
print(f"Date range: {int(base_features['year'].min())}–{int(base_features['year'].max())}")
print(f"Duplicate modeling keys: {int(base_features.duplicated(KEY_COLUMNS).sum()):,}")

## Generate EDA reports and visuals

The reusable EDA module produces distinct visuals for Gold integrity, missingness, temporal coverage, geographic coverage, numeric distributions, pairwise feature correlations, and a final findings summary.

In [ ]:
from eda_reporting import generate_eda_outputs  # Import reusable EDA reporting logic.

EDA_REPORTS_DIR = OUTPUTS_DIR / "eda_reports"  # Store reusable EDA tables.
EDA_VISUALS_DIR = OUTPUTS_DIR / "eda_visuals"  # Store reusable PNG visuals.

eda_outputs = generate_eda_outputs(
    base_features=base_features,
    reports_dir=EDA_REPORTS_DIR,
    visuals_dir=EDA_VISUALS_DIR,
)  # Generate all EDA tables and figures.

display(eda_outputs["findings_summary"])  # Keep the final EDA-to-modeling summary distinctly visible.

## Output inventory

These files can be copied into SharePoint and reused directly in FigJam.

In [ ]:
print("EDA REPORTS")
for path in sorted(EDA_REPORTS_DIR.glob("*.csv")):
    print(path)

print("\nEDA VISUALS")
for path in sorted(EDA_VISUALS_DIR.glob("*.png")):
    print(path)

print("\nEDA complete. Next lifecycle stage: restricted-target integration, feature engineering, temporal split, and model evaluation in Databricks.")